In [0]:

%run "../utils/config_loader"

In [0]:
# Cell 1 — Widget
dbutils.widgets.dropdown("environment", "dev", ["dev", "preprod", "prod"], "Environment")


In [0]:
# Cell 3 — Load config
config = load_config_from_widget()
print(f"🚀 Gold Pharmacy Pipeline - {config['environment'].upper()} Environment")
print(f"Status: Running gold transformation...")
print(f"📂 Reading from : {config['catalog']}.{config['schemas']['silver']}.pharmacy_clean")

# Cell 4 — Environment check
if config['environment'] == 'prod':
    print("⚠️  WARNING: Running on PROD! Make sure preprod is verified!")
elif config['environment'] == 'preprod':
    print("⚠️  WARNING: Running on PREPROD!")
else:
    print("✅ Running on DEV — safe to proceed!")

# Cell 5 — Imports
from pyspark.sql.functions import *

# Cell 6 — Read from silver
silver_table = f"{config['catalog']}.{config['schemas']['silver']}.pharmacy_clean"
df_silver = spark.table(silver_table)
print(f"\n📊 Silver rows: {df_silver.count()}")
display(df_silver)

# Cell 7 — dim_stores
dim_stores = df_silver \
    .select("store_id", "store_name", "region") \
    .dropDuplicates(["store_id"])

print(f"✅ dim_stores: {dim_stores.count()} rows")
display(dim_stores)

# Cell 8 — dim_drugs
dim_drugs = df_silver \
    .select("drug_id", "drug_name", "category", "unit_price") \
    .dropDuplicates(["drug_id"])

print(f"✅ dim_drugs: {dim_drugs.count()} rows")
display(dim_drugs)

# Cell 9 — dim_pharmacists
dim_pharmacists = df_silver \
    .select("pharmacist_id") \
    .dropDuplicates(["pharmacist_id"])

print(f"✅ dim_pharmacists: {dim_pharmacists.count()} rows")
display(dim_pharmacists)

# Cell 10 — fact_sales
fact_sales = df_silver \
    .select("sale_id", "store_id", "drug_id", "pharmacist_id",
            "quantity", "total_amount", "sale_date", "region",
            "is_high_value", "sale_priority", "is_prescription",
            "ingestion_timestamp")

print(f"✅ fact_sales: {fact_sales.count()} rows")
display(fact_sales)

# Cell 11 — Save gold tables
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{config['schemas']['gold']}")

gold_path = config['storage']['external']['gold']

dim_stores.write.format("delta").mode("overwrite") \
    .option("path", f"{gold_path}/dim_stores") \
    .saveAsTable(f"{config['catalog']}.{config['schemas']['gold']}.dim_stores")

dim_drugs.write.format("delta").mode("overwrite") \
    .option("path", f"{gold_path}/dim_drugs") \
    .saveAsTable(f"{config['catalog']}.{config['schemas']['gold']}.dim_drugs")

dim_pharmacists.write.format("delta").mode("overwrite") \
    .option("path", f"{gold_path}/dim_pharmacists") \
    .saveAsTable(f"{config['catalog']}.{config['schemas']['gold']}.dim_pharmacists")

fact_sales.write.format("delta").mode("overwrite") \
    .option("path", f"{gold_path}/fact_sales") \
    .saveAsTable(f"{config['catalog']}.{config['schemas']['gold']}.fact_sales")

print(f"\n=== GOLD COMPLETE ===")
print(f"🌍 Environment    : {config['environment'].upper()}")
print(f"✅ dim_stores     : {dim_stores.count()} rows")
print(f"✅ dim_drugs      : {dim_drugs.count()} rows")
print(f"✅ dim_pharmacists: {dim_pharmacists.count()} rows")
print(f"✅ fact_sales     : {fact_sales.count()} rows")
print(f"📂 Gold path      : {gold_path}")